# LightGBM Inference

This notebook downloads the registered LightGBM pipeline from W&B Model Registry, predicts on the Walmart competition test set, creates a Kaggle-ready submission, and logs the inference artifact to W&B.

The registered LightGBM bundle contains the fitted feature-engineering pipeline, selected feature list, and best fitted LightGBM model. Because the current LightGBM feature pipeline includes lag/rolling features, this inference notebook also loads historical train rows to provide sales history before transforming the raw test period.


In [ ]:
%pip install -q "lightgbm>=4,<5" "wandb>=0.19,<1" "pandas>=2.2,<3" "numpy>=1.26,<3" "cloudpickle>=3,<4" "matplotlib>=3.8,<4" "kaggle>=1.7,<2"


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print(f"Drive mount skipped: {exc}")


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import time
from pathlib import Path

import cloudpickle
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wandb

WANDB_ENTITY = "kende23-n-a"
WANDB_PROJECT = "Walmart-Recruiting---Store-Sales-Forecasting"
REGISTRY_ARTIFACT_URI = "wandb-registry-model/Walmart_LightGBM_Pipeline:champion"
PIPELINE_FILENAME = "lightgbm_best_pipeline.pkl"

DATA_DIR_CANDIDATES = [
    Path("/content/drive/MyDrive/walmart_competition_data"),
    Path("/content/drive/My Drive/walmart_competition_data"),
    Path("/content/walmart_competition_data"),
    Path("data"),
    Path("../../data"),
]
OUTPUT_DIR = Path("/content/artifacts/lightgbm_inference") if Path("/content").exists() else Path("artifacts/lightgbm_inference")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SUBMIT_TO_KAGGLE = False
KAGGLE_COMPETITION = "walmart-recruiting-store-sales-forecasting"
KAGGLE_MESSAGE = "W&B registered LightGBM pipeline"


In [ ]:
def resolve_data_dir(candidates):
    required = ["train.csv", "test.csv", "features.csv", "stores.csv"]
    for candidate in candidates:
        if all((candidate / filename).exists() for filename in required):
            return candidate
    raise FileNotFoundError(
        "Could not find Walmart data directory with train.csv, test.csv, features.csv, and stores.csv. "
        f"Checked: {[str(path) for path in candidates]}"
    )


DATA_DIR = resolve_data_dir(DATA_DIR_CANDIDATES)
print(f"Using data directory: {DATA_DIR}")


## Load raw competition data

`test.csv` remains the prediction target. `train.csv`, `features.csv`, and `stores.csv` are loaded only to reproduce the same feature columns expected by the registered LightGBM pipeline.


In [ ]:
train_raw = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["Date"])
test_raw = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["Date"])
features_raw = pd.read_csv(DATA_DIR / "features.csv", parse_dates=["Date"])
stores_raw = pd.read_csv(DATA_DIR / "stores.csv")

required_test_columns = ["Store", "Dept", "Date", "IsHoliday"]
if test_raw.columns.tolist() != required_test_columns:
    raise ValueError(f"Expected test columns {required_test_columns}, got {test_raw.columns.tolist()}")
if test_raw.duplicated(["Store", "Dept", "Date"]).any():
    raise ValueError("test.csv contains duplicate Store/Dept/Date rows")

train_merged = train_raw.merge(stores_raw, on="Store", how="left")
train_merged = train_merged.merge(
    features_raw,
    on=["Store", "Date", "IsHoliday"],
    how="left",
)

test_merged = test_raw.merge(stores_raw, on="Store", how="left")
test_merged = test_merged.merge(
    features_raw,
    on=["Store", "Date", "IsHoliday"],
    how="left",
)
test_merged["Weekly_Sales"] = np.nan

history_rows = train_merged.sort_values(["Date", "Store", "Dept"]).reset_index(drop=True)
test_rows_for_features = test_merged.sort_values(["Date", "Store", "Dept"]).reset_index(drop=False).rename(columns={"index": "original_test_index"})
combined_for_features = pd.concat(
    [
        history_rows.assign(__is_test=False, original_test_index=-1),
        test_rows_for_features.assign(__is_test=True),
    ],
    ignore_index=True,
    sort=False,
)

test_profile = {
    "test_rows": int(len(test_raw)),
    "history_rows": int(len(history_rows)),
    "test_weeks": int(test_raw["Date"].nunique()),
    "test_start": str(test_raw["Date"].min().date()),
    "test_end": str(test_raw["Date"].max().date()),
    "test_holiday_rows": int(test_raw["IsHoliday"].sum()),
}
display(pd.Series(test_profile, name="value").to_frame())
display(test_raw.head())


## Download registered LightGBM pipeline


In [ ]:
try:
    from google.colab import userdata
    wandb_api_key = userdata.get("WANDB_API_KEY")
except Exception:
    wandb_api_key = os.environ.get("WANDB_API_KEY")

wandb.login(key=wandb_api_key, relogin=False) if wandb_api_key else wandb.login()

run = wandb.init(
    entity=WANDB_ENTITY,
    project=WANDB_PROJECT,
    job_type="lightgbm_inference",
    name="LightGBM_Registry_Inference_Submission",
    tags=["lightgbm", "inference", "registry", "kaggle-submission"],
    config={
        "registry_artifact": REGISTRY_ARTIFACT_URI,
        "pipeline_filename": PIPELINE_FILENAME,
        "data_dir": str(DATA_DIR),
        "submit_to_kaggle": SUBMIT_TO_KAGGLE,
        **test_profile,
    },
)

model_artifact = run.use_artifact(REGISTRY_ARTIFACT_URI)
artifact_dir = Path(model_artifact.download(root=str(OUTPUT_DIR / "registry_model")))
pipeline_candidates = list(artifact_dir.rglob(PIPELINE_FILENAME))
if len(pipeline_candidates) != 1:
    raise FileNotFoundError(f"Expected one {PIPELINE_FILENAME}, found {pipeline_candidates}")

pipeline_path = pipeline_candidates[0]
with pipeline_path.open("rb") as file:
    pipeline = cloudpickle.load(file)

if not hasattr(pipeline, "feature_pipeline") or not hasattr(pipeline, "model"):
    raise TypeError("Registered artifact does not look like the expected LightGBM pipeline bundle.")

artifact_info = {
    "artifact_name": model_artifact.name,
    "artifact_version": model_artifact.version,
    "pipeline_path": str(pipeline_path),
    "selected_features": int(len(pipeline.selected_features)),
    "validation_weighted_mae": float(getattr(pipeline, "validation_weighted_mae", np.nan)),
}
display(pd.Series(artifact_info, name="value").to_frame())


## Transform features and predict

The current registered LightGBM feature pipeline expects historical `Weekly_Sales` to compute lag/rolling features. Therefore, the transformation is run on historical train rows plus test rows, then only transformed test rows are sent to the fitted model.


In [ ]:
started_at = time.perf_counter()
transformed_all = pipeline.feature_pipeline.transform(combined_for_features)
feature_seconds = time.perf_counter() - started_at

transformed_test = transformed_all.loc[combined_for_features["__is_test"].to_numpy()].copy()
if len(transformed_test) != len(test_raw):
    raise ValueError(f"Expected {len(test_raw)} transformed test rows, got {len(transformed_test)}")

missing_features = [feature for feature in pipeline.selected_features if feature not in transformed_test.columns]
if missing_features:
    raise ValueError(f"Missing selected features after transformation: {missing_features}")

X_test_selected = transformed_test[pipeline.selected_features].copy()
started_at = time.perf_counter()
predictions_sorted = pipeline.model.predict(X_test_selected)
prediction_seconds = time.perf_counter() - started_at

predictions_sorted = np.asarray(predictions_sorted, dtype="float64")
if predictions_sorted.shape != (len(test_raw),):
    raise ValueError(f"Prediction shape mismatch: {predictions_sorted.shape}")
if not np.isfinite(predictions_sorted).all():
    raise ValueError("Predictions contain non-finite values")

# Restore the original test.csv row order for Kaggle submission.
sorted_original_index = test_rows_for_features["original_test_index"].to_numpy()
predictions = np.empty_like(predictions_sorted)
predictions[sorted_original_index] = predictions_sorted
predictions = np.clip(predictions, 0.0, None)

prediction_stats = {
    "inference/feature_seconds": float(feature_seconds),
    "inference/prediction_seconds": float(prediction_seconds),
    "inference/submission_rows": int(len(predictions)),
    "inference/prediction_min": float(predictions.min()),
    "inference/prediction_mean": float(predictions.mean()),
    "inference/prediction_max": float(predictions.max()),
    "inference/prediction_std": float(predictions.std()),
}
run.log(prediction_stats)
run.summary.update(prediction_stats)
display(pd.Series(prediction_stats, name="value").to_frame())


## Create Kaggle submission


In [ ]:
submission = pd.DataFrame({
    "Id": (
        test_raw["Store"].astype(str)
        + "_"
        + test_raw["Dept"].astype(str)
        + "_"
        + test_raw["Date"].dt.strftime("%Y-%m-%d")
    ),
    "Weekly_Sales": predictions,
})

if submission["Id"].duplicated().any():
    raise ValueError("Submission contains duplicate Id values")
if submission["Weekly_Sales"].isna().any():
    raise ValueError("Submission contains missing predictions")

submission_path = OUTPUT_DIR / "submission_lightgbm_registry.csv"
manifest_path = OUTPUT_DIR / "lightgbm_inference_manifest.json"
submission.to_csv(submission_path, index=False)

manifest = {
    **artifact_info,
    **prediction_stats,
    "registry_artifact": REGISTRY_ARTIFACT_URI,
    "submission_path": str(submission_path),
    "created_at_utc": pd.Timestamp.utcnow().isoformat(),
    "competition": KAGGLE_COMPETITION,
}
manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

print(f"Submission saved to: {submission_path}")
display(submission.head())


## Log diagnostics and submission artifact


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(submission["Weekly_Sales"], bins=80)
ax.set_title("LightGBM registry submission prediction distribution")
ax.set_xlabel("Weekly_Sales")
ax.set_ylabel("Count")
plt.tight_layout()
hist_path = OUTPUT_DIR / "lightgbm_prediction_histogram.png"
fig.savefig(hist_path, dpi=160)
plt.show()

submission_artifact = wandb.Artifact(
    "lightgbm-kaggle-submission",
    type="submission",
    description="Kaggle submission generated from the W&B Registry LightGBM pipeline.",
    metadata=manifest,
)
submission_artifact.add_file(str(submission_path))
submission_artifact.add_file(str(manifest_path))
submission_artifact.add_file(str(hist_path))
run.log_artifact(submission_artifact, aliases=["latest"])

run.log({
    "inference/submission_preview": wandb.Table(dataframe=submission.head(2000)),
    "inference/prediction_histogram": wandb.Image(str(hist_path)),
})
for key, value in manifest.items():
    run.summary[key] = value


## Optional Kaggle upload

Set `SUBMIT_TO_KAGGLE = True` only when you intentionally want to submit. In Colab, put `KAGGLE_USERNAME` and `KAGGLE_KEY` in Secrets first.


In [ ]:
if SUBMIT_TO_KAGGLE:
    try:
        from google.colab import userdata
        kaggle_username = userdata.get("KAGGLE_USERNAME")
        kaggle_key = userdata.get("KAGGLE_KEY")
    except Exception:
        kaggle_username = os.environ.get("KAGGLE_USERNAME")
        kaggle_key = os.environ.get("KAGGLE_KEY")

    if not kaggle_username or not kaggle_key:
        raise RuntimeError("KAGGLE_USERNAME and KAGGLE_KEY are required for Kaggle upload.")

    kaggle_env = {
        **os.environ,
        "KAGGLE_USERNAME": kaggle_username,
        "KAGGLE_KEY": kaggle_key,
    }
    completed = subprocess.run(
        [
            "kaggle", "competitions", "submit",
            "-c", KAGGLE_COMPETITION,
            "-f", str(submission_path),
            "-m", KAGGLE_MESSAGE,
        ],
        check=True,
        capture_output=True,
        text=True,
        env=kaggle_env,
    )
    print(completed.stdout)
    run.summary["submission/kaggle_uploaded"] = True
else:
    print("Kaggle upload skipped. The submission CSV is ready for manual upload.")
    run.summary["submission/kaggle_uploaded"] = False


In [ ]:
run.finish()
print("Inference complete: registry LightGBM model -> test set -> submission artifact.")
